# RAG with LlamIndex

In [5]:
%load_ext autoreload
%autoreload 2

from datasets import load_dataset

from utils import *

import warnings
warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## About FinanceBench Data

* [Huggingface FinanceBench data][1]
  * [data dictionary][2]
    * `justification` is the expected answer to be compared with `answer`
    * `evidence` is the evidence of justification
  * JSONL data
    * Text are processed markdown text
    * Has metadata ready
  * Download `pdfs/` [here][2]

[1]:https://huggingface.co/datasets/PatronusAI/financebench
[2]:https://github.com/patronus-ai/financebench/tree/main

In [2]:
# Load FinanceBench dataset from Hugging Face
dataset = load_dataset("PatronusAI/financebench")

# Explore the dataset structure
print(dataset)
print(dataset['train'].column_names)
for k, v in dataset['train'][0].items():
    print(f"{k}: {v}")
    if k == 'evidence':
        print(v[0].keys())
    print()

DatasetDict({
    train: Dataset({
        features: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link'],
        num_rows: 150
    })
})
['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']
financebench_id: financebench_id_03029

company: 3M

doc_name: 3M_2018_10K

question_type: metrics-generated

question_reasoning: Information extraction

domain_question_num: None

question: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

answer: $1577.00

justification: The metric capital expenditures was directly extracted from

In [9]:
# Collect documents for LlamaIndex
documents = []

for split in dataset.keys():
    for item in dataset[split]:
        doc_path = 'pdfs/' + item.get('doc_name', None) + '.pdf'
        doc_content = load_pdf_content(doc_path)
        metadata = {}
        for key, value in item.items():
            if key not in ['document', 'question', 'answer']:
                metadata[key] = value
        
        doc = Document(text=doc_content, metadata=metadata)
        documents.append(doc)

        break

print(f"Loaded {len(documents)} documents from FinanceBench")

Loaded 1 documents from FinanceBench
